# 1장 실습: 왜 XAI인가

『설명가능한 인공지능(XAI)』 1장 실습 노트북입니다. 실습 저장소: https://github.com/neomoon40/xai-textbook

셀을 위에서부터 차례로 실행하세요. 뒤의 셀은 앞 셀이 만든 데이터와 모델을 씁니다.

## 0. 라이브러리 버전 맞추기 (Colab)

이 책의 결과는 Python 3.13, scikit-learn 1.9.1, pandas 3.0.5, NumPy 2.5.3에서 얻었습니다. Colab에 미리 설치된 버전은 이와 다르므로, **새 세션을 열 때마다** 아래 셀을 가장 먼저 실행합니다.

1. 아래 셀의 둘째 줄 맨 앞 `#`을 지우고 실행합니다.
2. 빨간 글씨로 `ERROR: pip's dependency resolver ...` 경고가 나올 수 있습니다. 마지막 줄에 `Successfully installed`가 나오면 설치는 끝난 것입니다.
3. **[런타임] → [세션 다시 시작]**을 누른 뒤, 이 셀은 건너뛰고 코드 1-1부터 실행합니다.

내 컴퓨터에서 부록 B의 requirements.txt로 이미 설치했다면 이 셀은 실행하지 않습니다.

In [ ]:
# Colab에서는 다음 줄의 #을 지우고 실행한다
# !pip install scikit-learn==1.9.1 pandas==3.0.5 numpy==2.5.3


### 실습 1. 환경 준비와 대출 심사 데이터 — 코드 1-1 실습 환경 확인

In [ ]:
import sys, sklearn, numpy, pandas as pd
print("Python", sys.version.split()[0])
print("scikit-learn", sklearn.__version__)
print("pandas", pd.__version__)
print("NumPy", numpy.__version__)


코드 1-1의 출력이 scikit-learn 1.9.1, pandas 3.0.5, NumPy 2.5.3이 아니면 위 설치 셀을 실행했는지, 세션을 다시 시작했는지 확인합니다.

### 코드 1-2 대출 심사 데이터 준비

In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

df = fetch_openml("credit-g", version=1, as_frame=True).frame
X = df.drop(columns="class")
y = (df["class"] == "good").astype(int)      # 1 = 승인, 0 = 거절
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)
print(X.shape, y.value_counts().to_dict())
print(X.iloc[0])


### 실습 2. 투명한 모델과 블랙박스 모델 비교 — 코드 1-3 세 모델의 성능 비교

In [ ]:
from sklearn.compose import make_column_transformer, make_column_selector
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

def prep(scale=False):
    # 범주형은 원-핫 인코딩, 수치형은 그대로(로지스틱 회귀만 표준화)
    cat = make_column_selector(dtype_include="category")
    num = make_column_selector(dtype_include="number")
    return make_column_transformer(
        (OneHotEncoder(handle_unknown="ignore"), cat),
        (StandardScaler() if scale else "passthrough", num),
        verbose_feature_names_out=False)

models = {
    "결정트리(깊이 3)": make_pipeline(
        prep(), DecisionTreeClassifier(max_depth=3, random_state=42)),
    "로지스틱 회귀": make_pipeline(
        prep(scale=True), LogisticRegression(max_iter=1000)),
    "랜덤 포레스트(300그루)": make_pipeline(
        prep(), RandomForestClassifier(n_estimators=300,
                                       random_state=42)),
}
print(f"모두 승인: 정확도 {y_test.mean():.3f}")
for name, m in models.items():
    m.fit(X_train, y_train)
    acc = accuracy_score(y_test, m.predict(X_test))
    print(f"{name}: 정확도 {acc:.3f}")


### 코드 1-4 결정트리 전체를 출력하기

In [ ]:
from sklearn.tree import export_text
tree_pipe = models["결정트리(깊이 3)"]
tree = tree_pipe[-1]
names = list(tree_pipe[0].get_feature_names_out())
print(export_text(tree, feature_names=names, decimals=2))

rf_pipe = models["랜덤 포레스트(300그루)"]
rf = rf_pipe[-1]
print("결정트리 노드 수:", tree.tree_.node_count)
print("랜덤 포레스트 전체 노드 수:",
      sum(e.tree_.node_count for e in rf.estimators_))


### 코드 1-5 신청자 A의 거절 이유 따라가기

In [ ]:
import numpy as np
both_reject = ((tree_pipe.predict(X_test) == 0)
               & (rf_pipe.predict(X_test) == 0))
a = X_test[both_reject].index[0]     # 두 모델이 모두 거절한 첫 신청자
x = X_test.loc[[a]]
print("신청자 A의 인덱스:", a)
print(x[["checking_status", "duration", "credit_amount",
         "purpose", "savings_status", "age"]].to_string())
print("트리 예측:", tree_pipe.predict(x)[0],
      f"/ 포레스트 승인 확률: {rf_pipe.predict_proba(x)[0, 1]:.2f}")

xt = tree_pipe[0].transform(x)
xt = xt.toarray() if hasattr(xt, "toarray") else np.asarray(xt)
for node in tree.decision_path(xt).indices:
    f = tree.tree_.feature[node]
    if f >= 0:                        # 잎이 아닌 노드만
        v, t = xt[0, f], tree.tree_.threshold[node]
        op = "<=" if v <= t else ">"
        print(f"  {names[f]} = {v:.2f} {op} {t:.2f}")


### 실습 3. 블랙박스를 밖에서 들여다보기 — 코드 1-6 순열 중요도로 랜덤 포레스트 들여다보기

In [ ]:
from sklearn.inspection import permutation_importance
r = permutation_importance(rf_pipe, X_test, y_test,
                           n_repeats=20, random_state=42)
imp = pd.Series(r.importances_mean, index=X.columns)
print(imp.sort_values(ascending=False).head(5).round(4))
